# Riemann data pipeline — consolidated 00–05

Single Colab pipeline. The environment stage is restart-safe: after the editable install, the runtime is terminated once; on the next Run all the first cell restores the repository working directory before any relative path is used.

Stages: 00 environment → integrity → 01 acquire → 02 describe → 03 unfold → 04 surrogates → 05 compare.

**Artifact rule:** existing artifacts are never overwritten. Existing raw/derived artifacts are verified against `data/manifest.json`. A missing derived artifact is created once, then hashed and added to the local manifest; any later mismatch is a hard error.

## 00 — Environment / Colab restart

Run all once. If the package is not installed from this checkout, this cell installs it and deliberately terminates the runtime. **Run all again after Colab reconnects.** The repository `cd` is performed on every run, including the post-restart run.


In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

REPO_BASE = Path("/content/nicht-riemann-data").resolve()
REPO_URL = "https://github.com/nicht-organization/nicht-riemann-data.git"

print("[00] pre-clone pwd:")
subprocess.run(["pwd"], check=True)
if not REPO_BASE.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_BASE)], check=True)
elif not (REPO_BASE / ".git").is_dir():
    raise RuntimeError(f"Path exists but is not a Git repository: {REPO_BASE}")

os.chdir(REPO_BASE)
print("[00] repo pwd:")
subprocess.run(["pwd"], check=True)
print("cwd:", Path.cwd())
print("python:", sys.version)
subprocess.run(["git", "status", "--short"], check=True)
subprocess.run(["git", "branch", "--show-current"], check=True)

package_spec = importlib.util.find_spec("nicht_riemann_data")
package_from_checkout = False
if package_spec is not None and package_spec.origin is not None:
    try:
        package_from_checkout = Path(package_spec.origin).resolve().is_relative_to(REPO_BASE / "src")
    except ValueError:
        package_from_checkout = False

if not package_from_checkout:
    print("Installing editable package from checkout; restarting runtime...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)
    print("[00] pwd immediately before os._exit(0):")
    subprocess.run(["pwd"], check=True)
    os._exit(0)

print("[00] pwd after restart/import path setup:")
subprocess.run(["pwd"], check=True)
from nicht_riemann_data.transforms import spacings, normalized_spacings
from nicht_riemann_data.diagnostics import describe

print("Package import: OK")


## Integrity helpers


In [ ]:
import hashlib
import json

MANIFEST_FILE = REPO_BASE / "data/manifest.json"
DATA_DIR = REPO_BASE / "data/raw"
DERIVED_DIR = REPO_BASE / "data/derived"

print("[integrity] pwd:")
subprocess.run(["pwd"], check=True)
print("[integrity] manifest:", MANIFEST_FILE)
print("[integrity] manifest exists:", MANIFEST_FILE.is_file())
assert MANIFEST_FILE.is_file(), f"Missing manifest: {MANIFEST_FILE}"
DATA_DIR.mkdir(parents=True, exist_ok=True)
DERIVED_DIR.mkdir(parents=True, exist_ok=True)

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def load_manifest() -> dict:
    with MANIFEST_FILE.open("r", encoding="utf-8") as f:
        value = json.load(f)
    assert isinstance(value, dict), "manifest root must be an object"
    return value

def save_manifest(manifest: dict) -> None:
    tmp = MANIFEST_FILE.with_suffix(".json.tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)
        f.write("\n")
    tmp.replace(MANIFEST_FILE)

manifest = load_manifest()
assert "datasets" in manifest and "odlyzko_zeros1" in manifest["datasets"]
print("manifest loaded: OK")


## 01 — Acquire

The raw dataset is required locally. It is never downloaded over an existing file.


In [ ]:
import numpy as np

print("[01] pwd:")
subprocess.run(["pwd"], check=True)
DATASET = "zeros1"
RAW_FILE = DATA_DIR / DATASET
raw_spec = manifest["datasets"]["odlyzko_zeros1"]
assert raw_spec["local_file"] == "data/raw/zeros1"
assert RAW_FILE.is_file(), f"Missing raw artifact: {RAW_FILE}"

actual_bytes = RAW_FILE.stat().st_size
actual_sha256 = sha256(RAW_FILE)
assert actual_bytes == raw_spec["raw_bytes"], f"Raw artifact size mismatch: {actual_bytes} != {raw_spec['raw_bytes']}"
assert actual_sha256 == raw_spec["sha256"], f"Raw artifact SHA-256 mismatch: {actual_sha256} != {raw_spec['sha256']}"

print(f"Verified raw artifact: {RAW_FILE}")
print("bytes:", actual_bytes)
print("SHA-256:", actual_sha256)
